# 5.2 DAS And Pointwise Strain

Distributed acoustic sensing (DAS) receivers are modeled as fiber channels that average strain over a gauge length. This tutorial compares three measurements on the same elastic model: a straight DAS fiber, a helical DAS fiber, and a colocated point strain receiver. The result cells plot gather differences and center-channel waveforms so the averaging behavior is visible.

By the end, you should be able to compare straight DAS, helical DAS, and point strain responses and explain how gauge-length averaging changes recorded traces.


## Design Notes

`ReceiverFiber` defines an averaged strain response along a channel. `L_gauge` is the physical gauge length and `n_gauge` is the number of integration samples used along that gauge. A straight fiber measures projected strain along the fiber direction. A helical fiber adds `radius` and `pitch`; the response is still a receiver component, but the solver evaluates the helical path over the gauge sample points.

| Receiver | Averaging path | What to compare |
| --- | --- | --- |
| Straight DAS | Straight gauge segment centered at each channel. | Baseline gauge-averaged strain. |
| Helical DAS | Helical path around the nominal fiber centerline. | Sensitivity change from helical winding. |
| Point strain node | Single receiver point, no gauge average. | Reference for how much gauge averaging smooths the signal. |

A point strain sensor is just a `ReceiverNode` with `field="strain"`. It is intentionally colocated with the DAS channels here so differences come from receiver response, not source or geometry changes.


## Imports


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import frequensolve as fs

u = fs.ureg


## Elastic Model For DAS Response

DAS is most naturally demonstrated in elastic physics because strain is a primary receiver quantity. The model and source are compact so the comparison runs quickly once a local solver is available.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="das_comparison",
    path="./scratch/tutorials/das",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="das_comparison",
    physics="elastic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(
    name="elastic",
    properties={"Vp": 2.4 * u.km / u.s, "Vs": 1.1 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3},
)
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model

sim += model.hex_mesh_generator([8, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim.mesh.set_source_grading(d0=0.02, d1=0.08, mult=2.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

model.plot("vp", figsize=(7, 3), aspect="equal")


## Straight DAS, Helical DAS, And Point Strain

The three receiver groups share coordinates, source, and component orientation. That makes the resulting trace differences attributable to receiver response rather than survey geometry. The helical example uses a small radius and pitch in model length units.


In [ ]:
straight_das = fs.ReceiverFiber(name="straight_das", L_gauge=0.01, n_gauge=5)
straight_das.add_component(name="eps_fiber", field="strain")

helical_das = fs.ReceiverFiber(name="helical_das", L_gauge=0.01, n_gauge=9, radius=0.002, pitch=0.05)
helical_das.add_component(name="eps_fiber", field="strain")

point_strain = fs.ReceiverNode(name="point_strain")
point_strain.add_component(name="eps_xx", field="strain_xx")

acq = fs.Acquisition()
acq.add_source_group(kind="vector", coords=[[0.5, 0.05]], direction=[0.0, 1.0])
coords = [[x, 0.05] for x in np.linspace(0.1, 0.9, 61)]
acq.add_receiver_group(name="straight_das", device=straight_das, coords=coords)
acq.add_receiver_group(name="helical_das", device=helical_das, coords=coords)
acq.add_receiver_group(name="point_strain", device=point_strain, coords=coords)
sim += acq

{
    group.name: [component.name for component in group.device.components]
    for group in acq.receiver_groups
}


## Run The DAS Comparison

The solver writes each receiver group separately. After the run, the group names in `traces.summary` should match the three groups above.


In [ ]:
sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

site = fs.LocalSite(shutdown_on_completion=True, verbose=True)
job = fs.TimeDomainJob(
    name="time_das",
    simulation=sim,
    f_min=0.0,
    f_max=30.0,
    T_max=0.9,
)
result = site.submit(job).wait()
traces = result.traces(upscale=4)
traces.summary


## Compare DAS Gather Responses

The straight and helical fibers both average strain over the gauge length, but their sampling paths differ. The point strain receiver samples the projected strain at the receiver location. The difference panels below are a quick quality-control view of how much the receiver response changes the data.


In [ ]:
wavelet = fs.RickerWavelet(f=12.0)
source = traces.sources("straight_das")[0]
straight = traces.td("straight_das", "eps_fiber", source, wavelet, upscale=4, T_max=0.9)
helical = traces.td("helical_das", "eps_fiber", source, wavelet, upscale=4, T_max=0.9)
point = traces.td("point_strain", "eps_xx", source, wavelet, upscale=4, T_max=0.9)

A = 2.0 * max(float(np.nanstd(np.real(arr.values))) for arr in [straight, helical, point])
fs.diff_gathers(
    straight,
    helical,
    A=A,
    cmap="gray",
    figsize=(12, 4),
    titles=("Straight DAS", "Helical DAS", "Helical - straight"),
)
fs.diff_gathers(
    point,
    straight,
    A=A,
    cmap="gray",
    figsize=(12, 4),
    titles=("Point strain", "Straight DAS", "Straight - point"),
)


## Overlay One Channel

A center-channel overlay makes the gauge averaging easier to see: phase shifts and amplitude smoothing are often clearer in line plots than in full gathers.


In [ ]:
center = len(coords) // 2
fig, ax = plt.subplots(figsize=(9, 4))
for label, gather in [("point strain", point), ("straight DAS", straight), ("helical DAS", helical)]:
    channel = gather.isel(receiver=center)
    ax.plot(channel["time"].values, np.real(channel.values), label=label)
ax.set_xlabel("Time")
ax.set_ylabel("Strain response")
ax.set_title(f"DAS response at channel {center + 1}")
ax.legend()
fig.tight_layout()


## Result Review Checklist

DAS review should compare responses, not just verify that a fiber object exported. The straight and helical gathers should be read as different receiver transfer functions applied to the same elastic wavefield, while the point strain trace is the unaveraged reference.

| Comparison | What it demonstrates |
| --- | --- |
| Straight DAS minus helical DAS | Sensitivity changes introduced by helical winding and gauge sampling. |
| Straight DAS minus point strain | Smoothing and phase effects from gauge-length averaging. |
| Center-channel overlay | Whether differences are coherent physical response changes rather than plotting scale artifacts. |
| Receiver definitions | Gauge length, sample count, radius, pitch, and component direction are the parameters to document in survey notes. |
